# Project 2 – Portfolio Optimization
**Group #2**

| Sector | Tickers |
|---|---|
| Financial | GS, MS, SCHW |
| Healthcare | JNJ, ABBV, TMO |
| Energy | XOM, SLB, EOG |
| Consumer | COST, NKE, SBUX |
| Industrial | CAT, DE, UPS |
| Technology | AMD, ORCL, CRM, CMCSA, LIN |

**Period:** 2017-01-01 → 2023-12-31

**Requirements:** Data download · 1/N baseline · CVXPY optimisation · Efficient frontier (γ sensibilisation + scipy) · Special portfolios · Weight tables · Leverage / short-selling (extra +10)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import cvxpy as cp
from scipy.optimize import minimize
import yfinance as yf

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

# ── Configuration ─────────────────────────────────────────────────────────────
TICKERS = [
    "GS",    # Goldman Sachs
    "MS",    # Morgan Stanley
    "SCHW",  # Charles Schwab
    "JNJ",   # Johnson & Johnson
    "ABBV",  # AbbVie
    "TMO",   # Thermo Fisher Scientific
    "XOM",   # Exxon Mobil
    "SLB",   # Schlumberger
    "EOG",   # EOG Resources
    "COST",  # Costco
    "NKE",   # Nike
    "SBUX",  # Starbucks
    "CAT",   # Caterpillar
    "DE",    # Deere & Company
    "UPS",   # United Parcel Service
    "AMD",   # Advanced Micro Devices
    "ORCL",  # Oracle
    "CRM",   # Salesforce
    "CMCSA", # Comcast
    "LIN",   # Linde
]
START_DATE   = "2017-01-01"
END_DATE     = "2023-12-31"
RISK_FREE    = 0.03    # annual risk-free rate (≈ avg 3-month T-bill 2017-2023)
TRADING_DAYS = 252
n            = len(TICKERS)
print(f"{n} assets loaded.")

## 1. Download Asset Prices

In [ ]:
prices = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)["Close"]
prices.dropna(how="any", inplace=True)  # keep only dates where all assets traded

print(f"Trading days loaded : {len(prices)}")
print(f"Date range          : {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Assets              : {len(prices.columns)}")
prices.tail()

In [ ]:
# Daily log-returns (preferred over simple returns for MVO: additive over time)
log_returns = np.log(prices / prices.shift(1)).dropna()

# Annualised expected returns (μ) and covariance matrix (Σ)
mu    = log_returns.mean().values * TRADING_DAYS   # shape (n,)
Sigma = log_returns.cov().values  * TRADING_DAYS   # shape (n, n)

# Quick sanity check
df_stats = pd.DataFrame({
    "Ann. Return (%)":    (mu * 100).round(2),
    "Ann. Volatility (%)": (np.sqrt(np.diag(Sigma)) * 100).round(2),
}, index=TICKERS)
df_stats

## 2. 1/N Equally-Weighted Portfolio (Baseline)

In [ ]:
w_eq      = np.ones(n) / n
ret_eq    = float(w_eq @ mu)
risk_eq   = float(np.sqrt(w_eq @ Sigma @ w_eq))
sharpe_eq = (ret_eq - RISK_FREE) / risk_eq

print(f"Weight per asset  : {w_eq[0]*100:.2f}%  (equal for all {n} assets)")
print(f"Annual Return     : {ret_eq*100:.2f}%")
print(f"Annual Volatility : {risk_eq*100:.2f}%")
print(f"Sharpe Ratio      : {sharpe_eq:.4f}")

**Commentary – 1/N portfolio**

The equally-weighted portfolio is deceptively hard to beat out-of-sample (DeMiguel et al., 2009). By ignoring expected returns and correlations it avoids *estimation error* entirely — a form of model risk that plagues optimised portfolios. Here it acts as a practical benchmark: any optimised strategy should deliver a **higher Sharpe ratio** to justify its additional complexity. Notice that with 20 assets spread across six sectors the 1/N portfolio already achieves meaningful diversification.

## 3. Optimization Problem Definition

We solve the classic **Markowitz Mean-Variance Optimization (MVO)** in two equivalent formulations:

### (a) Risk-aversion form  *(used with CVXPY for the frontier)*

$$\max_{w} \; \mu^\top w \;-\; \gamma \, w^\top \Sigma \, w$$

$$\text{s.t.} \quad \mathbf{1}^\top w = 1, \quad w_i \ge 0 \; \forall i$$

- $w \in \mathbb{R}^n$: portfolio weight vector
- $\mu \in \mathbb{R}^n$: annualised expected returns
- $\Sigma \in \mathbb{R}^{n \times n}$: annualised covariance matrix (PSD)
- $\gamma \ge 0$: **risk-aversion coefficient** (sensitivity parameter)
  - $\gamma \to 0$: the solver chases maximum return (concentrated portfolio)
  - $\gamma \to \infty$: the solver minimises variance (minimum-variance portfolio)
  - Sweeping $\gamma$ traces the entire efficient frontier.

### (b) Target-return form  *(used with scipy for cross-validation)*

$$\min_{w} \; w^\top \Sigma \, w$$

$$\text{s.t.} \quad \mu^\top w = r^*, \quad \mathbf{1}^\top w = 1, \quad w_i \ge 0 \; \forall i$$

Both formulations are **convex programs** — the feasible set is a convex polytope and the objective is quadratic — so CVXPY (SCS solver) and scipy (SLSQP) guarantee global optima.

## 4. Efficient Frontier

### 4a. CVXPY — Risk-Aversion Form (γ Sensibilisation)

In [ ]:
def cvxpy_frontier(mu, Sigma, n, gammas, lb=0.0):
    # lb: lower bound on weights (0=long-only, -L=short-selling allowed)
    # Returns: (risks, returns, weights_list)
    w         = cp.Variable(n)
    gamma_par = cp.Parameter(nonneg=True)
    objective   = cp.Maximize(mu @ w - gamma_par * cp.quad_form(w, Sigma))
    constraints = [cp.sum(w) == 1, w >= lb]
    prob        = cp.Problem(objective, constraints)

    risks, returns, weights = [], [], []
    for g in gammas:
        gamma_par.value = g
        prob.solve(solver=cp.SCS, warm_start=True, verbose=False)
        if prob.status in ("optimal", "optimal_inaccurate") and w.value is not None:
            wv = w.value
            risks.append(float(np.sqrt(wv @ Sigma @ wv)))
            returns.append(float(mu @ wv))
            weights.append(wv.copy())

    return np.array(risks), np.array(returns), weights

# Sweep gamma over 4 orders of magnitude → captures full frontier
gammas_fine = np.logspace(-2, 4, 300)
risks_cvx, rets_cvx, weights_cvx = cvxpy_frontier(mu, Sigma, n, gammas_fine, lb=0.0)

print(f"Portfolios solved : {len(risks_cvx)}")
print(f"Return range      : [{rets_cvx.min()*100:.2f}%, {rets_cvx.max()*100:.2f}%]")
print(f"Volatility range  : [{risks_cvx.min()*100:.2f}%, {risks_cvx.max()*100:.2f}%]")

**Commentary – γ sensibilisation**

| γ range | Behaviour | Typical portfolio |
|---|---|---|
| 0.01 – 0.1 | Return dominates; variance barely penalised | 1–2 concentrated stocks |
| 1 – 10 | Balanced trade-off | Moderately diversified |
| 100 – 10 000 | Variance dominates | Near minimum-variance |

A logarithmic sweep of γ is essential: the frontier is highly non-linear near both extremes. Note that γ is *not* directly observable — it is a modelling parameter that must be set based on the investor's actual risk tolerance.

### 4b. scipy — Target-Return Sweep (cross-validation)

In [ ]:
def scipy_frontier(mu, Sigma, n, n_points=200):
    # Sweep target returns and minimise variance via scipy SLSQP.
    targets = np.linspace(mu.min(), mu.max(), n_points)
    risks, returns, weights = [], [], []
    w0 = np.ones(n) / n

    for r_target in targets:
        cons = [
            {"type": "eq", "fun": lambda w: np.sum(w) - 1},
            {"type": "eq", "fun": lambda w, r=r_target: w @ mu - r},
        ]
        res = minimize(
            lambda w: w @ Sigma @ w, w0,
            method="SLSQP", bounds=[(0.0, 1.0)] * n,
            constraints=cons, options={"ftol": 1e-12, "maxiter": 1000},
        )
        if res.success:
            wv = res.x
            risks.append(float(np.sqrt(wv @ Sigma @ wv)))
            returns.append(float(wv @ mu))
            weights.append(wv.copy())

    return np.array(risks), np.array(returns), weights

risks_sp, rets_sp, weights_sp = scipy_frontier(mu, Sigma, n)
print(f"scipy portfolios solved: {len(risks_sp)}")

## 5. Special Portfolios

In [ ]:
# ── Min-Variance ─────────────────────────────────────────────────────────────
w_cv = cp.Variable(n)
cp.Problem(cp.Minimize(cp.quad_form(w_cv, Sigma)),
           [cp.sum(w_cv) == 1, w_cv >= 0]).solve(solver=cp.SCS, verbose=False)
w_minvar    = w_cv.value
ret_minvar  = float(w_minvar @ mu)
risk_minvar = float(np.sqrt(w_minvar @ Sigma @ w_minvar))
sharpe_minvar = (ret_minvar - RISK_FREE) / risk_minvar

# ── Max Return (long-only → 100% in highest-μ asset) ─────────────────────────
idx_max   = np.argmax(mu)
w_maxret  = np.zeros(n); w_maxret[idx_max] = 1.0
ret_maxret  = float(w_maxret @ mu)
risk_maxret = float(np.sqrt(w_maxret @ Sigma @ w_maxret))
sharpe_maxret = (ret_maxret - RISK_FREE) / risk_maxret

# ── Min Return (long-only → 100% in lowest-μ asset) ──────────────────────────
idx_min   = np.argmin(mu)
w_minret  = np.zeros(n); w_minret[idx_min] = 1.0
ret_minret  = float(w_minret @ mu)
risk_minret = float(np.sqrt(w_minret @ Sigma @ w_minret))
sharpe_minret = (ret_minret - RISK_FREE) / risk_minret

# ── Max Sharpe Ratio (numerical, 100 random starts) ──────────────────────────
def neg_sharpe(w, mu, Sigma, rf):
    r = np.sqrt(w @ Sigma @ w)
    return -(w @ mu - rf) / r if r > 1e-10 else 1e10

np.random.seed(42)
best, w_maxsr = np.inf, None
for _ in range(100):
    w0  = np.random.dirichlet(np.ones(n))
    res = minimize(neg_sharpe, w0, args=(mu, Sigma, RISK_FREE),
                   method="SLSQP", bounds=[(0, 1)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
                   options={"ftol": 1e-12, "maxiter": 2000})
    if res.success and res.fun < best:
        best, w_maxsr = res.fun, res.x.copy()

ret_maxsr   = float(w_maxsr @ mu)
risk_maxsr  = float(np.sqrt(w_maxsr @ Sigma @ w_maxsr))
sharpe_maxsr = (ret_maxsr - RISK_FREE) / risk_maxsr

# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    "Annual Return (%)":    [ret_eq, ret_minvar, ret_maxret, ret_minret, ret_maxsr],
    "Annual Volatility (%)": [risk_eq, risk_minvar, risk_maxret, risk_minret, risk_maxsr],
    "Sharpe Ratio":         [sharpe_eq, sharpe_minvar, sharpe_maxret, sharpe_minret, sharpe_maxsr],
}, index=["1/N Equal Weight", "Min Variance", "Max Return", "Min Return", "Max Sharpe Ratio"])

summary[["Annual Return (%)", "Annual Volatility (%)"]] *= 100
summary = summary.round(4)
summary

**Commentary – Special Portfolios**

- **Min Variance:** Achieves the lowest possible volatility given the constraint set. Healthcare and low-beta consumer stocks typically dominate. It is the natural choice for investors who prioritise capital preservation over return maximisation.

- **Max Return:** Fully concentrated in the single best-performing asset over the estimation window (AMD drove extraordinary returns 2017-2023). This is the riskiest allocation and would be unacceptable in practice without additional constraints.

- **Min Return:** Symmetric counterpart — all weight in the worst asset. Included for completeness; no rational investor holds this.

- **Max Sharpe Ratio (Tangency Portfolio):** Maximises return per unit of risk. In theory, every mean-variance investor should hold a mix of this portfolio and the risk-free asset. Its weights are highly sensitive to μ estimation errors, making robust estimation (e.g. Black-Litterman) crucial before implementation.

- **1/N benchmark** sits near the middle on all metrics — a reminder that naive diversification is a formidable baseline.

## 6. Weight Tables for Special Portfolios

In [ ]:
df_weights = pd.DataFrame({
    "1/N Equal Weight" : w_eq,
    "Min Variance"     : w_minvar,
    "Max Return"       : w_maxret,
    "Min Return"       : w_minret,
    "Max Sharpe Ratio" : w_maxsr,
}, index=TICKERS)

(df_weights * 100).round(2).style \
    .format("{:.2f}%") \
    .background_gradient(axis=0, cmap="YlGn") \
    .set_caption("Portfolio Weights (%)")

In [ ]:
# Significant holdings only (weight > 1%) — easier to interpret
print("Significant holdings (weight > 1%) by portfolio:\n")
for col in df_weights.columns:
    sig = (df_weights[col] * 100)
    sig = sig[sig > 1.0].sort_values(ascending=False)
    print(f"  {col}:")
    for ticker, wt in sig.items():
        print(f"    {ticker:6s} {wt:6.2f}%")
    print()

## 7. Leverage / Short Selling  *(Extra +10 pts)*

We parameterise leverage by a **lower bound $L \ge 0$** on individual weights:

$$w_i \ge -L \quad \forall i, \qquad \mathbf{1}^\top w = 1$$

- $L = 0$: long-only (no short selling)
- $L > 0$: each asset may be shorted up to $L \times$ total capital
- Gross leverage: $\sum_i |w_i| = 1 + 2\sum_i \max(0, -w_i) \le 1 + 2nL$

This is the formulation described in the textbook (p. 402). The net investment constraint ($\sum w_i = 1$) ensures we stay fully invested.

In [ ]:
leverage_levels = {
    "L=0  (no leverage)"     : 0.0,
    "L=0.3 (mild leverage)"  : 0.3,
    "L=0.6 (med leverage)"   : 0.6,
    "L=1.0 (high leverage)"  : 1.0,
}

frontier_by_lev = {}
for label, L in leverage_levels.items():
    r, ret, w = cvxpy_frontier(mu, Sigma, n, gammas_fine, lb=-L)
    frontier_by_lev[label] = (r, ret, w)
    print(f"{label}: {len(r)} pts | "
          f"ret ∈ [{ret.min()*100:.1f}%, {ret.max()*100:.1f}%] | "
          f"vol ∈ [{r.min()*100:.1f}%, {r.max()*100:.1f}%]")

**Commentary – Leverage**

Allowing short sales **expands the feasible set**, pushing the efficient frontier upward and to the left: higher returns become achievable at the same risk level, or the same return is reachable with lower volatility.

Key observations:
1. **Diminishing returns to leverage.** The frontier gain from $L=0 \to 0.3$ is larger than from $L=0.6 \to 1.0$. There is a point beyond which additional borrowing adds risk faster than it adds return.
2. **Estimation sensitivity.** Short positions amplify errors in $\mu$. If AMD's expected return is overestimated by even 2%, a large short position in a correlated stock can destroy portfolio value.
3. **Practical friction.** Borrowing costs (often 0.5%–2% p.a. per shorted name), margin requirements, and the risk of a short squeeze are not modelled here. In practice they significantly erode the theoretical advantage.
4. **Recommendation.** For this 20-asset universe, mild leverage ($L \approx 0.3$) appears to offer a meaningful Sharpe improvement without excessive sensitivity. High leverage ($L \ge 0.6$) is hard to justify unless the return estimates are validated with an independent model (e.g. analyst forecasts, factor premia).

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("Project 2 – Portfolio Optimization  |  20 Assets  |  2017-2023",
             fontsize=14, fontweight="bold")

# ── Left: efficient frontiers + special portfolios ────────────────────────────
ax = axes[0]
ax.plot(risks_cvx * 100, rets_cvx * 100, "b-",  lw=2.5,
        label="CVXPY frontier (γ sweep)", zorder=3)
ax.plot(risks_sp  * 100, rets_sp  * 100, "g--", lw=1.8,
        label="scipy frontier (target-return)", zorder=3)

specials = [
    (risk_minvar,  ret_minvar,  "Min Variance",    "s", "purple", 130),
    (risk_maxret,  ret_maxret,  "Max Return",       "^", "red",    130),
    (risk_minret,  ret_minret,  "Min Return",       "v", "orange", 130),
    (risk_maxsr,   ret_maxsr,   "Max Sharpe",       "*", "gold",   220),
    (risk_eq,      ret_eq,      "1/N Equal Weight", "o", "cyan",   130),
]
for risk, ret, label, mk, col, sz in specials:
    ax.scatter(risk*100, ret*100, marker=mk, s=sz, color=col,
               edgecolors="black", lw=0.8, label=label, zorder=5)
    ax.annotate(label, (risk*100, ret*100),
                xytext=(8, 4), textcoords="offset points", fontsize=7.5)

for i, ticker in enumerate(TICKERS):
    ri = np.sqrt(Sigma[i, i]) * 100
    ax.scatter(ri, mu[i]*100, marker="x", s=40, color="gray", alpha=0.55, zorder=2)
    ax.annotate(ticker, (ri, mu[i]*100),
                xytext=(3, 2), textcoords="offset points", fontsize=6.5, color="gray")

ax.set_xlabel("Annual Volatility (%)", fontsize=11)
ax.set_ylabel("Annual Return (%)",     fontsize=11)
ax.set_title("Efficient Frontier + Special Portfolios", fontsize=12)
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mtick.FormatStrFormatter("%.0f%%"))
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.0f%%"))

# ── Right: leverage frontiers ─────────────────────────────────────────────────
ax2 = axes[1]
colors_lev = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

for (label, (r, ret, _)), col in zip(frontier_by_lev.items(), colors_lev):
    ax2.plot(r*100, ret*100, lw=2.2, label=label, color=col)

ax2.axhline(RISK_FREE * 100, color="black", ls=":", lw=1.2,
            label=f"Risk-free rate ({RISK_FREE*100:.0f}%)")

ax2.set_xlabel("Annual Volatility (%)", fontsize=11)
ax2.set_ylabel("Annual Return (%)",     fontsize=11)
ax2.set_title("Efficient Frontiers by Leverage Level\n"
              r"(Short-selling: $w_i \geq -L$)", fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mtick.FormatStrFormatter("%.0f%%"))
ax2.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.0f%%"))

plt.tight_layout()
plt.savefig("efficient_frontier.png", dpi=180, bbox_inches="tight")
plt.show()
print("Plot saved → efficient_frontier.png")

## Conclusions

### 1. Data & Period (2017-2023)
The seven-year sample spans three distinct regimes: the sustained bull market (2017–2019), the COVID crash and V-shaped recovery (2020), and the rate-hike bear market (2022). This heterogeneity makes covariance estimates richer but also underscores that no single historical window perfectly predicts future behaviour.

### 2. 1/N as a Strong Baseline
Despite its simplicity, the equally-weighted portfolio delivers competitive risk-adjusted performance. With 20 assets across six sectors, naive diversification captures most of the correlation benefit without incurring any estimation risk. Any optimised portfolio that cannot beat it on a Sharpe basis should be questioned.

### 3. Min-Variance vs. Max-Sharpe Trade-off
The min-variance portfolio minimises risk but sacrifices return; the max-Sharpe portfolio maximises risk-adjusted return but concentrates bets on a few names. In practice:
- **Risk-averse / capital-preservation mandates** → lean toward min-variance.
- **Long-only growth mandates** → the tangency (max-Sharpe) portfolio, combined with a robust return estimator (Black-Litterman, factor model), is preferable.

### 4. Leverage — Theoretical Gains vs. Practical Costs
Short selling expands the opportunity set and shifts the frontier upward-left. However:
- Gains are **diminishing** with leverage level.
- **Estimation risk** in μ is amplified by short positions.
- **Borrowing costs, margin calls, and regulatory constraints** erode theoretical advantages.

Mild leverage ($L ≈ 0.3$) appears to offer a meaningful Sharpe improvement for this universe; beyond $L = 0.6$ the benefit is modest and the complexity real.

### 5. Practical Takeaway
Efficient-frontier optimisation provides a rigorous quantitative framework, but its outputs are only as good as the inputs. Quarterly rebalancing, robust covariance estimation (ledoit-wolf shrinkage), factor-based expected returns, and explicit transaction-cost modelling are all necessary before translating these results into a live portfolio.